In [1]:
import math 
import numpy as np
import pandas as pd

In [2]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [ ]:
class HiddenMarkov:
    def __init__(self, init_probs, trans_probs, emit_probs):
        self.states = tuple(init_probs)
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs

    def validation(self, observation):
        if not observation:
            raise ValueError("Observation cannot be empty")

        alphabet = set()

        for emissions in self.emit_probs.values():
            alphabet.update(emissions.keys())

            if round(sum(emissions.values()), 6) != 1:
                raise ValueError("Emission probabilities must sum to 1")

        if round(sum(self.init_probs.values()), 6) != 1:
            raise ValueError("Initial probabilities must sum to 1")

        for transitions in self.trans_probs.values():
            if round(sum(transitions.values()), 6) != 1:
                raise ValueError("Transition probabilities must sum to 1")

        for symbol in observation:
            if symbol not in alphabet:
                raise ValueError(f"Invalid observation: {symbol}")
        
    def initialization(self, observation):

        # create score table and previous state tables
        scores = []
        previous = []

        for symbol in observation:
            score_column = {}
            previous_column = {}

            for state in self.states:
                score_column[state] = float("-inf")
                previous_column[state] = None
            
            scores.append(score_column)
            previous.append(previous_column)

        for state in self.states:
            scores[0][state] = (
                self._log(self.init_probs[state]) 
                + self._log(self.emit_probs[state][observation[0]])
            )
        
        return scores, previous
    
    def _log(self, probability):
        if probability == 0:
            return -math.inf
        return math.log(probability)
    
        # Recursion 
        # FOR each position t after the first
        #     FOR each current state
        #         FOR each possible preceding state
        #             candidate =
        #                 score[t - 1][preceding state]
        #                 + log_transition[preceding state][current state]
        #                 + log_emission[current state][observations[t]]

        #             IF candidate is better than score[t][current state]
        #                 store candidate as the new score[position][current]
        #                 store preceding state as the traceback pointer - preceding[position][current]

    def _recursion(self, observation):

        scores, previous = self.initialization(observation)

        for position in range(1, len(observation)):
            for current in self.states:
                for preceding in self.states:

                    candidate = (
                        scores[position - 1][preceding]
                        + self._log(self.trans_probs[preceding][current])
                        + self._log(self.emit_probs[current][observation[position]])
                    )

                    if candidate > scores[position][current]:
                        scores[position][current] = candidate
                        previous[position][current] = preceding

        return scores, previous
        

    def traceback(self, observation):

        scores, previous = self._recursion(observation)

        # Find the state with the highest final score
        final = max(scores[-1], key=scores[-1].get)
        best_log = scores[-1][final]

        # Start with the final state
        path = [final]

        # Follow the previous states backwards
        for position in range(len(observation) - 1, 0, -1):
            final = previous[position][final]
            path.append(final)

        # Reverse to get the correct order
        path.reverse()

        return path, best_log


In [10]:
model = HiddenMarkov(init_probs, trans_probs, emit_probs)
model.validation(obs)

print("Validation passed!")

Validation passed!


In [11]:
scores, previous = model.initialization(obs)

print("First observation:", obs[0])

for state in model.states:
    print(state, scores[0][state], previous[0][state])

First observation: G
I -2.525728644308255 None
G -1.83258146374831 None


In [12]:
scores, previous = model.initialization(obs)

print("Observations:", len(obs))
print("Score rows:", len(scores))
print("Previous rows:", len(previous))

Observations: 9
Score rows: 9
Previous rows: 9


In [13]:
model._recursion(obs)

([{'I': -2.525728644308255, 'G': -1.83258146374831},
  {'I': -3.7986943201211427, 'G': -3.5473798918402366},
  {'I': -5.07165999593403, 'G': -5.262178319932163},
  {'I': -7.730920032866807, 'G': -6.571511639915925},
  {'I': -9.003885708679695, 'G': -8.28631006800785},
  {'I': -11.663145745612473, 'G': -9.595643387991613},
  {'I': -12.814519212859812, 'G': -11.31044181608354},
  {'I': -15.47377924979259, 'G': -12.619775136067302},
  {'I': -17.224945322055394, 'G': -13.929108456051065}],
 [{'I': None, 'G': None},
  {'I': 'I', 'G': 'G'},
  {'I': 'I', 'G': 'G'},
  {'I': 'I', 'G': 'G'},
  {'I': 'I', 'G': 'G'},
  {'I': 'I', 'G': 'G'},
  {'I': 'G', 'G': 'G'},
  {'I': 'I', 'G': 'G'},
  {'I': 'G', 'G': 'G'}])

In [8]:
scores, previous = model._recursion(obs)

print("Final scores:", scores[-1])
print("Final backpointers:", previous[-1])

Final scores: {'I': -17.224945322055394, 'G': -13.929108456051065}
Final backpointers: {'I': 'G', 'G': 'G'}


In [16]:
model.traceback(obs)


(['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G'], -13.929108456051065)